# ColorCorrect: chart to calibrated images

Fit a chart photograph, adjust FFC/GC/WB/CC settings, and apply the correction to
other photographs taken under the same conditions. Install the SDK wheel and
Jupyter, then set `COLORCORRECT_APP_ROOT`, `COLORCORRECT_SAMPLES`, and
`COLORCORRECT_OUTPUT_DIR` before starting the kernel. Results are saved in a new
subdirectory for each run. With the current source engine, Auto chart and
white-plane detection try an eligible GPU and recover to CPU when needed;
fitting uses the CPU. Published 0.2.1 packages still require GPU inference.

See [notebook setup](README.md), [configuration reference](../docs/api/README.md#configuration-reference),
[ColorCorrectionPipeline](https://pypi.org/project/ColorCorrectionPipeline/), and
[the original study](https://doi.org/10.1002/ppj2.70067).

The workflow also covers CC-only validation, saved models, reference profiles,
and cumulative stage images.

## 1. Locate the installed SDK and inputs

Select the kernel from the environment where you installed the wheel. Set the
application, sample-photo, and output directories explicitly.

In [ ]:
import json, os, sys, tempfile
from pathlib import Path
import colorcorrect
from colorcorrect import (
    ApplyOptions, CalibrationOptions, Client, ColorCorrection, Config, FFCOptions,
    NeuralNetworkOptions, ReferenceProfile,
)

def required_directory(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} to an explicit directory before starting the kernel")
    path = Path(value).expanduser().resolve()
    if not path.is_dir():
        raise FileNotFoundError(f"{variable}: {path}")
    return path

app = required_directory("COLORCORRECT_APP_ROOT")
samples = required_directory("COLORCORRECT_SAMPLES")
output_root = required_directory("COLORCORRECT_OUTPUT_DIR")
module_path = Path(colorcorrect.__file__).resolve()
assert module_path.is_relative_to(Path(sys.prefix).resolve()), "Install the wheel in this kernel environment; do not import from a checkout"
chart, white = samples / "ColorCard_Big.JPG", samples / "Blank.JPG"
scenes = [samples / "Sample_3572.JPG", samples / "Sample_3573.JPG"]
assert all(path.is_file() for path in [chart, white, *scenes]), "The public sample set is incomplete"
out = Path(tempfile.mkdtemp(prefix="tutorial-", dir=output_root))
print("SDK:", colorcorrect.__version__, module_path)
print("Native app:", app, "\nArtifacts:", out)

## 2. Edit the pipeline

These are frozen dataclasses. `replace` makes a new configuration. Config stage switches override switches in nested `CalibrationOptions`. GC and WB are fitted automatically; their controls are on/off. The `degree` below configures the spatial FFC surface, not gamma. A manual white crop can replace `None` with `"x1,y1,x2,y2"` in white-image pixels. Choose coordinates for your image.

In [ ]:
from dataclasses import replace
config = Config(
    save_path=out, do_ffc=True, do_gc=True, do_wb=True, do_cc=True,
    options=CalibrationOptions(
        model="rpcc2", ffc=FFCOptions(method="pls", bins=32, smooth=5, degree=3),
        crop_rect=None, cross_validate=True,
    ),
)
without_gc = replace(config, do_gc=False)  # Use for a separate new fit.
without_wb = replace(config, do_wb=False)
polynomial = replace(config, options=replace(config.options, model="linear", degree=2))
print("Current fit:", config.calibration_options())

## 3. Detect and fit

The blank photograph supplies illumination only. Reuse this detection only for this unchanged chart photograph. Keep fitted-chart error separate from CC-only held-out validation: FFC/GC/WB are fitted using the full selected chart, so this is not independent full-pipeline validation.

In [ ]:
with ColorCorrection(config, root=app, backend="auto", timeout=900) as cc:
    worker = cc.client.engine.executable.resolve()
    assert worker.is_relative_to(app), "Worker must belong to the selected application"
    detection = cc.client.detect(chart)
    calibration = cc.run(Image=chart, White_Image=white, detection=detection)
    model_path = calibration.path
    report = calibration.metrics
    assert report is not None and report.get("do_ffc") is True, calibration.warnings
    assert calibration.corrected_image and calibration.corrected_image.is_file()
    corrected = calibration.corrected_image
    print("Worker:", worker)
    print("Actual fit backends:", calibration.backends.actual)
    print("Fitted-chart DeltaE00:", report.get("delta_e_2000"))
    print("CC-only held-out validation:", report.get("validation_delta_e_2000"))
    print("Warnings:", calibration.warnings)
    result = cc.predict_images(images=scenes, options=ApplyOptions(output_format="jpg"))
    assert len(result.outputs) == 2 and all(path.is_file() for path in result.outputs)
    print("Outputs:", result.outputs)
    print("Actual apply execution:", result.backends.actual, result.backends.diagnostics)

## 4. Reload, inspect stages, and export

A new process/client loads the JSON model. FFC inherits the stored white path, content check, crop, and settings. Keep that asset accessible. Explicit `ApplyOptions(white=...)` selects a replacement; ensure it matches the geometry. Raw preview calls are an advanced Engine escape hatch: pass FFC rendering settings explicitly there. A cube LUT excludes spatial FFC.

In [ ]:
with ColorCorrection(Config(save_path=out / "reloaded"), root=app, timeout=900) as cc:
    loaded = cc.load_model(model_path)
    reused = cc.predict_image(scenes[0], options=ApplyOptions(output_format="jpg"))
    assert reused.images[0].ok and reused.outputs[0].is_file()
    # Applying a fitted model with WB disabled is an ablation, not a refit.
    no_wb = cc.predict_image(scenes[0], options=ApplyOptions(do_wb=False, write_output=False))
    assert no_wb.images[0].ok and not no_wb.outputs
    preview_dir = out / "stages"
    for stage in ("ffc", "gc", "wb", "cc"):
        cc.client.engine.preview(
            model_path, chart, stage, preview_dir,
            do_ffc=True, white=white, ffc=config.options.ffc.to_dict(), fast_mode=True,
        )
    lut = loaded.export_lut(out / "correction.cube", size=33)
    assert lut.path.is_file()
    print("LUT:", lut.path, lut.warnings)

## 5. Import measured references

Set `COLORCORRECT_REFERENCE_CSV` to a complete Classic/Passport Lab D50 CSV
(24 triplets). Built-in references require no argument. The detected ID below
selects one compatible chart; multiple charts need one profile assignment per
selected ID. Supply reference measurements for your chart: importing a CSV maps
the values to cells but does not measure them.

In [ ]:
reference_csv = os.environ.get("COLORCORRECT_REFERENCE_CSV")
if reference_csv:
    reference = ReferenceProfile.from_csv(
        reference_csv, chart="classic", color_space="lab", white_point="d50", order="row-major",
    )
    with Client(root=app, timeout=900) as client:
        custom = client.calibrate(
            chart, detection=detection, reference=reference, chart_ids=[detection.charts[0].id],
            output_dir=out / "custom", options=CalibrationOptions(write_output=False),
        )
        assert custom.path.is_file() and custom.metrics.get("reference_context")
        print("Custom reference model:", custom.path)
else:
    print("Custom-reference exercise not executed: set COLORCORRECT_REFERENCE_CSV to run it.")

## 6. Compare the stage images

PNG thumbnails display the original photograph and cumulative correction stages.
Intermediate previews can differ from the final Fast-mode approximation. Use the
chart metrics and photographs captured separately to assess correction quality.

In [ ]:
from io import BytesIO
from PIL import Image as PillowImage
from IPython.display import Image, Markdown, display

def show_thumbnail(path, caption):
    assert path.is_file(), path
    with PillowImage.open(path) as original:
        thumb = original.convert("RGB")
        thumb.thumbnail((640, 420))
        buffer = BytesIO()
        thumb.save(buffer, format="PNG")
    display(Markdown("**" + caption + "**"), Image(data=buffer.getvalue()))

for path, caption in [(chart, "Original"), (preview_dir / "ffc.png", "After spatial FFC"),
                      (preview_dir / "gc.png", "After FFC + GC"), (preview_dir / "wb.png", "After FFC + GC + WB"),
                      (corrected, "After FFC + GC + WB + CC")]:
    show_thumbnail(path, caption)

## Try another configuration

Change one stage switch with `replace`, refit, and compare the same photograph.
Read the CC-only validation score alongside the fitted-chart score; use a
separate capture to assess the full pipeline. Keep image encoding settings
consistent with the stored pixels. The legacy `backend="cpu"` selects CPU
correction with Auto detection. To force CPU detection, use
`backend="cpu-inference"` with a worker advertising `cpu_inference_v1`.

In [ ]:
print("Training mean after:", report["delta_e_2000"].get("mean_after"))
print("CC-only validation:", report.get("validation_delta_e_2000"))
print("Warnings:", report.get("fit_warnings", []))
print("Durable output directory:", out)
(out / "tutorial-result.json").write_text(json.dumps({"ok": True, "model": str(model_path),
    "worker": str(worker), "kernel_interpreter": sys.executable, "sdk_module": str(module_path), "output": str(out),
    "fit_backends": {"actual": calibration.backends.actual, "diagnostics": calibration.backends.diagnostics},
    "apply_backends": {"actual": result.backends.actual, "diagnostics": result.backends.diagnostics},
    "custom_reference_executed": bool(reference_csv)}, indent=2), encoding="utf-8")